# 5.5 Acht mensen, en hoe ze typen

Eén claim, helemaal doorgelicht via het raster van [05.1](05.1-relationships%20jeffrey.ipynb):

> **Mensen zijn beter herkenbaar aan *hoe* ze schrijven dan aan *waarover* ze schrijven.**

Mechanisme: aannemelijk, en het voorspelt iets specifieks — gewoonten zouden stabiel moeten blijven terwijl het onderwerp van gesprek verschuift. Bewijs: wat dit notebook opbouwt. Replicatie: er zit een gat van vier jaar in het corpus om dit op te controleren.

De acht drukste personen op `#ubuntu-uk`. Actieregels (`/me`) worden weggelaten: die hebben een andere grammatica en zouden een te makkelijke weggever zijn.

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import (
    CountValues,
    Filter,
    Head,
    Pipeline,
    RegexFeature,
    TransformBase,
)
from goad_toolkit.visualizer import (
    Annotate,
    BarPlot,
    GroupedBarPlot,
    HeatmapPlot,
    HighlightCategory,
    PlotSettings,
    ScatterPlot,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from scripts.pipelines import build_irc_pipeline
from wa_analyzer.data import load_showcase

msgs = build_irc_pipeline().apply(load_showcase("ubuntu_irc"))
uk = (
    Pipeline()
    .add(Filter, expr="channel == '#ubuntu-uk' and not is_action")
    .add(RegexFeature, name="caps", column="message", pattern=r"^[A-Z]",
         feature="starts_upper", mode="has")
    .apply(msgs)
)
uk["length"] = uk.message.str.len()
uk["night"] = uk.hh.between(0, 5)

top8 = Pipeline().add(CountValues, column="author").add(Head, n=8).apply(uk)
people = uk[uk.author.isin(top8.author)].copy()

baseline = top8.n.iloc[0] / top8.n.sum()
print(top8.to_string(index=False))
print(f"\n{len(people):,} messages, {people.author.nunique()} authors")
print(f"guessing the most frequent author every time = {baseline:.1%}")

In [ ]:
people.head()

## 5.5.1 De classifier is een instrument, niet het resultaat

Fit een model dat één bericht leest en de auteur ervan aanwijst. **De nauwkeurigheid (accuracy) is een poort, geen bevinding** — het vertelt je enkel of er überhaupt iets is om naar te kijken. Wat het model heeft *geleerd* is de echte bevinding, en dat zit verstopt in de gewichten (*weights*).

In [ ]:
train_msgs, test_msgs = train_test_split(
    people, test_size=0.25, random_state=42, stratify=people.author)

words = TfidfVectorizer(min_df=5, sublinear_tf=True)
one_liner = LogisticRegression(max_iter=1000, C=5)
one_liner.fit(words.fit_transform(train_msgs.message), train_msgs.author)

one_message_accuracy = one_liner.score(words.transform(test_msgs.message), test_msgs.author)
print(f"{len(words.vocabulary_):,} features, one message at a time")
print(f"accuracy {one_message_accuracy:.1%}  against a {baseline:.1%} baseline")

vocabulary = np.array(words.get_feature_names_out())
for i, author in enumerate(one_liner.classes_):
    strongest = vocabulary[np.argsort(one_liner.coef_[i])[::-1][:8]]
    print(f"{author:12s} {', '.join(strongest)}")

Twee keer de baseline is een drempel die comfortabel wordt gehaald, maar nog lang geen eindbestemming — de helft van de berichten wordt nog steeds aan de verkeerde persoon toegeschreven. Bekijk de rijen, en ze zijn niet allemaal van dezelfde aard:

- `ali1234` — `n900`, `qml`, `pidgin`, `mythtv`. **Onderwerp van gesprek.**
- `MartijnVdS` — `mungbean`, `dimpy`, `neuro`. **Bijnamen van andere mensen.** Helemaal geen schrijfgewoonte; een sociale positie.
- `zmoylan-pi` — `ireland`, `dublin`. **Locatie.**
- `daftykins` — `0o`, `xd`, `hrmm`. **Typegewoonten.**
- `popey` — `dont`, `didnt`, `thats`. **Typegewoonten** — specifiek: ontbrekende apostrofs.

Alleen die laatste twee betreffen de claim. De andere zijn een onderwerp-detector vermomd als stylometrie, en een onderwerp-detector faalt op het moment dat iemand van onderwerp verandert — wat exact de situatie is waarvoor een auteurschapsmodel bedoeld is.

> **De verificatiestap, en de gewoonte om over te nemen:** wanneer een model werkt, lees dan de gewichten uit en > vraag *wat elk gewicht feitelijk voorstelt*. Een model kan gelijk hebben om een reden die niet generaliseert, > en de accuracy zal je dat nooit vertellen.

Stop dus met het gebruiken van het model om de vingerafdruk te vinden. Meet de gewoonten direct, en laat het model later terugkeren als controle.

## 5.5.2 Vier gewoonten, gemeten

Niets hier heeft machine learning nodig. Vijf `RegexFeature` stappen en een `groupby` — letterlijk dezelfde transformatie die les 1 op URL's richtte, nu gericht op smileys en apostrofs. De ene niet-regex gewoonte is gratis: de `mentions`-stap uit les 1 extraheerde al tot wie een bericht zich richtte, dus "begint met iemands nick" is simpelweg of die extractie beet had.

Begin met smileys, waar drie manieren bestaan om hetzelfde te typen.

In [ ]:
habits = (
    Pipeline()
    .add(RegexFeature, name="nosed", column="message",
         pattern=r"[:;=]-[)DPp(\]]", feature="nosed", mode="count")
    .add(RegexFeature, name="noseless", column="message",
         pattern=r"[:;=][)DPp(\]]|\bXD\b", feature="noseless", mode="count")
    .add(RegexFeature, name="smiley", column="message",
         pattern=r"[☺☻☹㋛]", feature="unicode_smiley", mode="count")
    .add(RegexFeature, name="apos_drop", column="message",
         pattern=r"(?i)\b(?:dont|doesnt|didnt|cant|wont|isnt|im|thats|its|ive|youre)\b",
         feature="apos_drop", mode="count")
    .add(RegexFeature, name="apos_keep", column="message",
         pattern=r"(?i)\b(?:don't|doesn't|didn't|can't|won't|isn't|i'm|that's|it's|i've|you're)\b",
         feature="apos_keep", mode="count")
    .apply(people)
)
habits["addresses"] = habits.addressed_to.notna()

per_author = habits.groupby("author")
smileys = per_author[["nosed", "noseless", "unicode_smiley"]].sum()
dialect = smileys.div(smileys.sum(axis=1), axis=0) * 100
dialect.columns = pd.Index(["nosed :-)", "noseless :)", "unicode ☺"])

dialects = PlotSettings(
    figsize=(11, 4),
    title="Three ways to write a smiley, and nobody mixes them",
    xlabel="",
    ylabel="share of that author's smileys (%)",
    xtick_rotation=30,
)
fig, ax = GroupedBarPlot(dialects).plot(
    data=dialect.reset_index().melt(id_vars="author", var_name="dialect", value_name="share"),
    x="author", y="share", hue="dialect",
    palette={"nosed :-)": "crimson", "noseless :)": "#bbbbbb", "unicode ☺": "steelblue"},
)
_ = ax.legend(title="", loc="upper left", bbox_to_anchor=(1.01, 1))

Drie dialecten, en **niemand mengt ze**. `zmoylan-pi` en `diddledan` typen de neus bijna altijd, de andere zes vrijwel nooit — er zit niemand tussenin. Dat is niet hoe een gewoonte er gewoonlijk uitziet. De meeste meetbare verschillen tussen mensen zijn gradaties; dit is een **discrete keuze**, eenmalig gemaakt en vijf jaar lang volgehouden, door mensen die elkaars berichten elke dag lazen. `popey` vormt in zijn eentje het derde dialect: ☺, een karakter dat de anderen nooit intypen.

De overige gewoonten — apostrofs, hoofdletters, iemand aanspreken, 's nachts posten — gaan in één vingerafdruktabel, getekend als een heatmap zodat de acht rijen in één oogopslag te vergelijken zijn.

In [ ]:
fingerprint = pd.DataFrame({
    "nosed :-)": dialect["nosed :-)"],
    "unicode ☺": dialect["unicode ☺"],
    "no apostrophe": per_author.apos_drop.sum()
    / (per_author.apos_drop.sum() + per_author.apos_keep.sum()) * 100,
    "starts capital": per_author.starts_upper.mean() * 100,
    "addresses": per_author.addresses.mean() * 100,
    "00-05h": per_author.night.mean() * 100,
})

marks = PlotSettings(figsize=(9, 4.5), title="Eight fingerprints (% of that author's messages)",  # ty: ignore[invalid-argument-type]
                     xlabel="", ylabel="")
fig, ax = HeatmapPlot(marks).plot(data=fingerprint, annot=True, fmt=".0f", cmap="rocket_r")

Elke kolom heeft dezelfde vorm als de smileys: een paar mensen aan het ene uiterste en alle anderen geclusterd aan het andere. `popey` en `foobarry` laten een derde van hun apostrofs weg, alle anderen vrijwel geen. `daftykins` doet 15% van zijn interacties tussen middernacht en 05:00; `foobarry` heeft in vijf jaar tijd exact nul keer in dat venster gepost. `MartijnVdS` opent de helft van zijn berichten met iemands nick — wat de classifier vond, maar nu juist geïnterpreteerd: niet dat hij *anders schrijft*, maar dat zijn rol in het kanaal bestaat uit het beantwoorden van mensen.

Het kanaalgemiddelde verbergt dit allemaal. Er bestaat niet zoiets als een typische `#ubuntu-uk` auteur.

## 5.5.3 Waarom de classifier slechts 47% haalde, terwijl de gewoonten zo uitgesproken zijn

Beide zijn tegelijkertijd waar, en de reden is de eenheid van analyse.

In [ ]:
has_smiley = habits[["nosed", "noseless", "unicode_smiley"]].sum(axis=1) > 0
has_contraction = habits[["apos_drop", "apos_keep"]].sum(axis=1) > 0
print(f"messages containing a smiley       {has_smiley.mean():.1%}")
print(f"messages containing a contraction  {has_contraction.mean():.1%}")
print(f"messages containing neither        {(~has_smiley & ~has_contraction).mean():.1%}")

**Driekwart van de berichten bevat helemaal geen vingerafdruk.** De gewoonten zijn nagenoeg deterministisch *wanneer ze verschijnen*, maar de meeste losse regels zijn gewoon `ok`, `thanks`, `brb`. Een model dat één enkele regel leest, heeft meestal geen enkel aanknopingspunt, dus 47% is hoe een sterk signaal eruitziet wanneer het zeldzaam (*sparse*) is.

De oplossing is ophouden met vragen stellen over één enkel bericht. Voeg vijftig berichten per persoon samen tot een blok en vraag het opnieuw — en aangezien "wat telt als één rij" een beslissing is die verantwoording vereist, is het een eigen transformatie, hier gedefinieerd waar ze gelezen kan worden. Vervolgens twee modellen op dezelfde blokken: eentje die elk woord kent, en eentje die tien gewoonten kent en verder niets.

In [ ]:
class BlockMessages(TransformBase):
    """Pool each author's shuffled messages into fixed-size blocks: habit means, text joined."""

    def transform(self, data: pd.DataFrame, features: list[str], size: int = 50,
                  seed: int = 0) -> pd.DataFrame:
        shuffled = data.sample(frac=1, random_state=seed)
        shuffled["block"] = shuffled.groupby("author").cumcount() // size
        grouped = shuffled.groupby(["author", "block"])
        blocks = grouped[features].mean()
        blocks["text"] = grouped.message.apply(" ".join)
        return blocks[grouped.size() == size].reset_index()


FEATURES = ["nosed", "noseless", "unicode_smiley", "apos_drop", "apos_keep",
            "starts_upper", "addresses", "night", "length", "n_question"]

blocks = Pipeline().add(BlockMessages, features=FEATURES, size=50).apply(habits)
train, test = train_test_split(blocks, test_size=0.25, random_state=42, stratify=blocks.author)

vec = TfidfVectorizer(min_df=3, sublinear_tf=True)
vocabulary_model = LogisticRegression(max_iter=1000, C=5)
vocabulary_model.fit(vec.fit_transform(train.text), train.author)

scaler = StandardScaler().fit(train[FEATURES])
habit_model = LogisticRegression(max_iter=2000)
habit_model.fit(scaler.transform(train[FEATURES]), train.author)

scores = pd.DataFrame({
    "model": ["baseline", "1 message, words", "50 messages, words", "50 messages, habits"],
    "accuracy": [
        baseline,
        one_message_accuracy,
        vocabulary_model.score(vec.transform(test.text), test.author),
        habit_model.score(scaler.transform(test[FEATURES]), test.author),
    ],
})
gate = PlotSettings(
    figsize=(9, 4),
    title=f"{len(blocks):,} blocks of 50 messages: ten numbers do what {len(vec.vocabulary_):,} words do",
    xlabel="",
    ylabel="accuracy on held-out data",
    base_color="#cccccc",
    highlight_color="crimson",
)
bars = BarPlot(gate)
fig, ax = bars.plot(data=scores, x="model", y="accuracy", color=gate.base_color)
_ = bars.plot_on(HighlightCategory(gate), categories=["50 messages, habits"])
print(scores.round(3).to_string(index=False))

**Tien getallen doen wat zeventienduizend woorden doen.** Beide blokmodellen zitten tegen het plafond aan, en het model dat niets van vocabulaire weet — geen onderwerpen, geen bijnamen, geen plaatsnamen — zit binnen één procentpunt van het model dat alles weet. Dat is de claim, aangetoond: identiteit zit in de gewoonten, niet in het onderwerp. Er was volume voor nodig om het te zien, wat de eerlijke kanttekening is die in dezelfde zin thuishoort.

En omdat er tien features zijn in plaats van zeventienduizend, zijn de gewichten leesbaar: elke rij is een persoon, in de eigen woorden van het model.

In [ ]:
weights = pd.DataFrame(habit_model.coef_, index=habit_model.classes_, columns=pd.Index(FEATURES))
weights.round(1)

`MartijnVdS` — **adresseringen**, al het andere bijna nul: de rol, opnieuw. `daftykins` — **neusloos** en **nacht**: typt `:)` en is wakker om 03:00. `foobarry` — **nacht**, sterk negatief: grotendeels geïdentificeerd door er *nooit* te zijn. `popey` — unicode, weggelaten apostrofs, kort. Vergelijk dat met de tf-idf modelrijen in §5.5.1: dezelfde taak, dezelfde data, en eentje ervan kun je hardop voorlezen.

## 5.5.4 Pijler drie: houdt het stand als niemand kijkt?

Het corpus beslaat vijf jaar. Fit niets — meet simpelweg dezelfde gewoonten in 2013–2014 en opnieuw in 2016–2017, en kijk of mensen nog steeds zichzelf zijn. Vier gewoonten, één paneel per gewoonte: een auteur op de diagonaal is dezelfde persoon in beide helften.

In [ ]:
early = habits[habits.date.dt.year <= 2014]
late = habits[habits.date.dt.year >= 2016]
print(f"early {len(early):,} messages (2013-2014), late {len(late):,} (2016-2017)")

checks = [("addresses by nick (%)", "addresses", "mean", 100),
          ("starts with a capital (%)", "starts_upper", "mean", 100),
          ("median message length", "length", "median", 1),
          ("asks a question (%)", "n_question", "mean", 100)]

pairs = {}
for label, column, how, scale in checks:
    pair = pd.concat([early.groupby("author")[column].agg(how) * scale,
                      late.groupby("author")[column].agg(how) * scale], axis=1).dropna()
    pair.columns = pd.Index(["2013-2014", "2016-2017"])
    pairs[label] = pair

stable = PlotSettings(
    figsize=(11, 9),
    title="The same eight people, four years apart",
    subplot_titles=[f"{label}\nspearman {pair['2013-2014'].corr(pair['2016-2017'], method='spearman'):.2f}"
                    for label, pair in pairs.items()],
    max_cols=2,
)
host = ScatterPlot(stable)
fig, axes = host.create_figure(n_plots=4)
for ax, (label, pair) in zip(axes, pairs.items()):
    host.plot_on_axes(ScatterPlot(stable), ax, data=pair, x="2013-2014", y="2016-2017",
                      color="steelblue", s=60)
    lo, hi = pair.min().min(), pair.max().max()
    ax.plot([lo, hi], [lo, hi], color="lightgrey", linestyle="--", linewidth=1)
    for author, row in pair.iterrows():
        host.plot_on_axes(Annotate(stable), ax, text=str(author), xy=(row["2013-2014"], row["2016-2017"]),
                          xytext=(row["2013-2014"], row["2016-2017"]), arrow=False, fontsize=8)

Drie van de vier panelen plaatsen iedereen dicht bij de diagonaal — Spearman 0,93 tot 0,98: de rangorde van acht mensen op een typegewoonte is onveranderd na een onderbreking van vier jaar, op berichten waar niemand aan dacht toen de gewoonte ontstond. Pijler drie, glashelder.

Het vierde paneel is het meest leerzame. **Het aantal vragen is geen karaktereigenschap** — de punten liggen verspreid van de lijn af, Spearman 0,55. Of je vragen stelt hangt af van of je op dat moment een probleem hebt, en dat verandert; het verschuift mensen in de rangschikking terwijl hun typegewoonten hen vasthouden. Een bevinding die met haar eigen tegenvoorbeeld komt is geloofwaardiger dan eentje die dat niet doet, omdat het aantoont dat de meting ook 'nee' had kunnen zeggen.

## 5.5.5 En nu het bezwaar

Haal het eerlijk door het raster en het landt linksboven: mechanisme, bewijs, replicatie. Maar er is één zin die je niet kunt opschrijven:

**Er zijn hier maar acht mensen.**

Het bewijs bestaat uit 218.000 berichten, en elke toets hierboven is exact om die reden uitgevoerd op auteursniveau. Maar `n = 8`, en de claim — *mensen worden geïdentificeerd aan hoe ze typen* — gaat over mensen in het algemeen. Een Spearman van 0,98 over acht punten blijft een Spearman over acht punten. Dit is de meest verleidelijke valkuil van de cursus en daarom de beste plek om het te benoemen: **veel data lost een kleine n op het niveau van je claim niet op.** Pseudoreplicatie uit les 2, opduikend aan het einde van het notebook dat de hele rit zo voorzichtig is geweest.

Wat er nodig is om dit op te lossen: meer auteurs. De gewoonten zijn goedkoop te meten — elke cel hierboven draait op iedereen met dertig berichten — dus de eerlijke volgende stap is de stabiliteitscorrelaties opnieuw berekenen over elke auteur met voldoende geschiedenis in beide perioden, en *dát* getal rapporteren in plaats van dit.

> **Wat overeind blijft, geformuleerd zoals het gerapporteerd moet worden:** *"Onder de acht meest actieve `#ubuntu-uk` auteurs classificeren handmatig gekozen typegewoonten blokken van 50 berichten met ruim 99% nauwkeurigheid, gelijkwaardig aan een bag-of-words met 17.000 features, en de rangorde in gewoonten blijft behouden over een onderbreking van vier jaar (Spearman 0,93–0,98) terwijl het aantal vragen dat niet doet (0,55). Of dit standhoudt voor minder actieve auteurs is niet getoetst."*

Elke bijzin in die zin doet werk, en geen ervan bevat het woord "significant".

## Jouw beurt

Jouw chat bevat mensen. Kies twee gewoonten die je kunt uitdrukken in een regex — een smiley-dialect, een stopwoordje, een taalwisseling, een apostrof — en meet ze per auteur met de bovenstaande pipeline. Beantwoord vervolgens de twee vragen die dit notebook stelde: kan een model dat alleen die gewoonten kent mensen van elkaar onderscheiden op blokken berichten, en blijft de gewoonte van elke persoon behouden in de eerste en tweede helft van de chat? Rapporteer de n op auteursniveau, wat die ook mag zijn.